# News Topic Discovery — EDA and training walkthrough

This notebook is a thin, readable layer over the code in `src/`. It does not
duplicate any logic: every step calls the same functions the pipeline uses,
so what you see here is what actually produced `artifacts/`.

**Fixed training data** lives in `data/train_snapshot.csv`. Everything below is
the offline stage only — the live service loads the saved artifacts and never
re-fits.

In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from src.config import ARTIFACT_DIR, SNAPSHOT_CSV, SNAPSHOT_META
from src.predict import ArtifactBundle
from src.preprocess import clean_titles

df = pd.read_csv(SNAPSHOT_CSV)
meta = json.loads(SNAPSHOT_META.read_text(encoding="utf-8"))
print(f"{len(df):,} frozen headlines")
print(f"source        : {meta['source']}")
print(f"fetched at    : {meta['fetched_at_utc']}")
print(f"sha256        : {meta['sha256_train_csv'][:32]}...")
df[["created_at", "title", "points"]].head()

## What does a headline look like after cleaning?

`clean_title` strips URLs, `Show HN:`-style prefixes, years and punctuation.
The same function runs at serving time — a mismatch there would silently
degrade every prediction.

In [ ]:
samples = df["title"].head(5).tolist()
for raw, clean in zip(samples, clean_titles(samples)):
    print(f"raw  : {raw}")
    print(f"clean: {clean}\n")

lengths = pd.Series([len(t.split()) for t in clean_titles(df["title"]) if t])
lengths.describe()

In [ ]:
tokens = Counter()
for title in clean_titles(df["title"]):
    tokens.update(title.split())

pd.DataFrame(tokens.most_common(20), columns=["term", "count"])

## The fitted model

`python -m src.train` wrote these numbers. They are read straight from
`artifacts/metrics.json` rather than recomputed, so the notebook cannot drift
away from what is actually deployed.

In [ ]:
metrics = json.loads((ARTIFACT_DIR / "metrics.json").read_text(encoding="utf-8"))

keys = [
    "n_documents", "n_features", "svd_components", "svd_variance_kept",
    "k", "silhouette", "davies_bouldin", "calinski_harabasz",
    "novelty_threshold",
]
pd.DataFrame({k: [metrics[k]] for k in keys}).T.rename(columns={0: "value"})

In [ ]:
sweep = pd.Series(metrics["silhouette_sweep"], dtype=float).sort_index(key=lambda i: i.astype(int))
sweep.index = sweep.index.astype(int)
sweep.rename("silhouette")

In [ ]:
topics = pd.DataFrame(json.loads((ARTIFACT_DIR / "topics.json").read_text(encoding="utf-8")))
topics["keywords"] = topics["keywords"].str.join(", ")
topics.sort_values("size", ascending=False).reset_index(drop=True)

## Using the frozen model on new text

This is exactly the call the API makes. Note that `distance` is always exposed,
and headlines far from every centroid are marked `is_novel` instead of being
silently forced into a topic.

In [ ]:
bundle = ArtifactBundle.load()

incoming = [
    "Nvidia stock surges after earnings report",
    "Rust compiler gets faster incremental builds",
    "A recipe for sourdough bread with rye flour",
    "",
]

pd.DataFrame(bundle.predict_many(incoming), index=incoming)

In [ ]:
holdout = metrics.get("holdout", {})
print(f"held-out headlines : {holdout.get('n')}")
print(f"assigned           : {holdout.get('assigned_rate', 0):.1%}")
print(f"flagged novel      : {holdout.get('novel_rate', 0):.1%}")
print(f"inference speed    : {holdout.get('ms_per_item')} ms / headline")
pd.DataFrame(holdout.get("top_topics", []))

## Takeaways

1. Headlines are short (~8 words), so the representation is necessarily lossy:
   30 SVD components keep only ~7 % of TF-IDF variance and silhouette is ~0.24.
2. Frequency filtering (`max_df=0.05`) and a small domain stoplist removed the
   generic words that otherwise name every cluster.
3. The model generalises to unseen headlines (96 % assigned) but keeps an
   honest escape hatch: 9 % are flagged novel rather than guessed.
4. Inference costs ~0.06 ms per headline, so the live poller is limited by the
   network, not by the model.